### Source Tables:
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit` — patient visits

### Strategy:
- Filter Active = TRUE, exclude VisitStatus = 'CAN' (canceled)
- Map TypeCode to visit_detail_concept_id:
  - Ambulatory → 9202 (Outpatient Visit)
  - Outpatient → 9202 (Outpatient Visit)
  - Inpatient → 9201 (Inpatient Visit)
  - Emergency → 9203 (Emergency Room Visit)
- Start date = AdmitDtm, End date = COALESCE(DischargeDtm, CloseDtm, AdmitDtm)
- visit_detail_type_concept_id = 32817 (EHR)

### Notes:
- This notebook depends on source_to_person being populated for allscripts_scm
- provider_id, care_site_id, visit_occurrence_id will be NULL until those mappings exist
- VisitStatus values: ADM (admitted), PRE (pre-admit), DSC (discharged), CAN (canceled), CLS (closed)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS _exponent.omop_silver.visit_detail (
  visit_detail_concept_id INT,
  visit_detail_start_date DATE,
  visit_detail_start_datetime TIMESTAMP,
  visit_detail_end_date DATE,
  visit_detail_end_datetime TIMESTAMP,
  visit_detail_type_concept_id INT,
  visit_detail_source_value STRING,
  visit_detail_source_concept_id INT,
  admitted_from_concept_id INT,
  admitted_from_source_value STRING,
  discharged_to_concept_id INT,
  discharged_to_source_value STRING,
  person_source_value STRING,
  provider_source_value STRING,
  care_site_source_value STRING,
  visit_occurrence_source_value STRING,
  visit_detail_source_key STRING COMMENT 'Unique key for MERGE matching',
  source_system STRING,
  last_mod_tsp TIMESTAMP
)
USING DELTA
COMMENT 'Silver layer for OMOP visit_detail';

In [0]:
%sql
-- Create silver_visit_detail temp view for SCM
CREATE OR REPLACE TEMPORARY VIEW silver_visit_detail AS
SELECT
  -- Visit detail concept based on TypeCode
  CASE
    WHEN v.TypeCode = 'Inpatient' THEN 9201
    WHEN v.TypeCode IN ('Ambulatory', 'Outpatient') THEN 9202
    WHEN v.TypeCode = 'Emergency' THEN 9203
    ELSE 0
  END AS visit_detail_concept_id,

  -- Start date
  DATE(v.AdmitDtm) AS visit_detail_start_date,
  v.AdmitDtm AS visit_detail_start_datetime,

  -- End date
  DATE(COALESCE(v.DischargeDtm, v.CloseDtm, v.AdmitDtm)) AS visit_detail_end_date,
  COALESCE(v.DischargeDtm, v.CloseDtm, v.AdmitDtm) AS visit_detail_end_datetime,

  -- Type
  32817 AS visit_detail_type_concept_id,

  -- Source values
  v.TypeCode AS visit_detail_source_value,
  0 AS visit_detail_source_concept_id,

  -- Admitted from / Discharged to
  0 AS admitted_from_concept_id,
  NULL AS admitted_from_source_value,
  0 AS discharged_to_concept_id,
  v.DischargeDisposition AS discharged_to_source_value,

  -- FK source values
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(v.ClientGUID AS STRING)) AS person_source_value,
  NULL AS provider_source_value,
  NULL AS care_site_source_value,
  NULL AS visit_occurrence_source_value,

  -- Unique key
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3clientvisit', 'GUID', CAST(v.GUID AS STRING)) AS visit_detail_source_key,
  'allscripts_scm' AS source_system

FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit v

INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(v.ClientGUID AS STRING))
  AND stp.active_flag = TRUE

WHERE v.Active = TRUE
  AND v.VisitStatus != 'CAN'
  AND v.ClientGUID IS NOT NULL
  AND v.AdmitDtm IS NOT NULL

In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_visit_detail LIMIT 10

In [0]:
# %sql
# -- Validation
# SELECT
#   COUNT(*) AS total,
#   COUNT(DISTINCT person_source_value) AS distinct_patients,
#   SUM(CASE WHEN visit_detail_concept_id = 9201 THEN 1 ELSE 0 END) AS inpatient,
#   SUM(CASE WHEN visit_detail_concept_id = 9202 THEN 1 ELSE 0 END) AS outpatient,
#   SUM(CASE WHEN visit_detail_concept_id = 9203 THEN 1 ELSE 0 END) AS emergency,
#   SUM(CASE WHEN visit_detail_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped,
#   MIN(visit_detail_start_date) AS min_date,
#   MAX(visit_detail_start_date) AS max_date
# FROM silver_visit_detail

In [0]:
%sql
MERGE INTO _exponent.omop_silver.visit_detail AS t
USING silver_visit_detail AS s
ON t.visit_detail_source_key = s.visit_detail_source_key

WHEN MATCHED AND (
     NOT (t.visit_detail_concept_id <=> s.visit_detail_concept_id)
  OR NOT (t.visit_detail_start_date <=> s.visit_detail_start_date)
  OR NOT (t.visit_detail_start_datetime <=> s.visit_detail_start_datetime)
  OR NOT (t.visit_detail_end_date <=> s.visit_detail_end_date)
  OR NOT (t.visit_detail_end_datetime <=> s.visit_detail_end_datetime)
  OR NOT (t.visit_detail_type_concept_id <=> s.visit_detail_type_concept_id)
  OR NOT (t.visit_detail_source_value <=> s.visit_detail_source_value)
  OR NOT (t.visit_detail_source_concept_id <=> s.visit_detail_source_concept_id)
  OR NOT (t.admitted_from_concept_id <=> s.admitted_from_concept_id)
  OR NOT (t.admitted_from_source_value <=> s.admitted_from_source_value)
  OR NOT (t.discharged_to_concept_id <=> s.discharged_to_concept_id)
  OR NOT (t.discharged_to_source_value <=> s.discharged_to_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.visit_detail_concept_id        = s.visit_detail_concept_id,
  t.visit_detail_start_date        = s.visit_detail_start_date,
  t.visit_detail_start_datetime    = s.visit_detail_start_datetime,
  t.visit_detail_end_date          = s.visit_detail_end_date,
  t.visit_detail_end_datetime      = s.visit_detail_end_datetime,
  t.visit_detail_type_concept_id   = s.visit_detail_type_concept_id,
  t.visit_detail_source_value      = s.visit_detail_source_value,
  t.visit_detail_source_concept_id = s.visit_detail_source_concept_id,
  t.admitted_from_concept_id       = s.admitted_from_concept_id,
  t.admitted_from_source_value     = s.admitted_from_source_value,
  t.discharged_to_concept_id       = s.discharged_to_concept_id,
  t.discharged_to_source_value     = s.discharged_to_source_value,
  t.person_source_value            = s.person_source_value,
  t.provider_source_value          = s.provider_source_value,
  t.care_site_source_value         = s.care_site_source_value,
  t.visit_occurrence_source_value  = s.visit_occurrence_source_value,
  t.source_system                  = s.source_system,
  t.last_mod_tsp                   = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  visit_detail_source_value,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  person_source_value,
  provider_source_value,
  care_site_source_value,
  visit_occurrence_source_value,
  visit_detail_source_key,
  source_system,
  last_mod_tsp
)
VALUES (
  s.visit_detail_concept_id,
  s.visit_detail_start_date,
  s.visit_detail_start_datetime,
  s.visit_detail_end_date,
  s.visit_detail_end_datetime,
  s.visit_detail_type_concept_id,
  s.visit_detail_source_value,
  s.visit_detail_source_concept_id,
  s.admitted_from_concept_id,
  s.admitted_from_source_value,
  s.discharged_to_concept_id,
  s.discharged_to_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.care_site_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_key,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.visit_detail
# WHERE source_system = 'allscripts_scm'
# LIMIT 10

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS _exponent.omop_mapping.source_to_visit_detail (
#   visit_detail_id BIGINT GENERATED ALWAYS AS IDENTITY,
#   source_system STRING,
#   visit_detail_source_value STRING,
#   active_flag BOOLEAN,
#   created_tsp TIMESTAMP,
#   last_mod_tsp TIMESTAMP,
#   merge_id STRING,
#   merge_reason STRING
# )
# USING DELTA
# COMMENT 'Mapping table for visit_detail IDs';

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_detail (
    source_system,
    visit_detail_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.visit_detail_source_key,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, visit_detail_source_key, last_mod_tsp
    FROM _exponent.omop_silver.visit_detail
    WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visit_detail x
  ON s.visit_detail_source_key = x.visit_detail_source_value;

In [0]:
%sql
-- MERGE INTO _exponent.omop.visit_detail AS gold
MERGE INTO _exponent.omop_scm.visit_detail AS gold
USING (
  SELECT
    svd.visit_detail_id,
    stp.person_id,
    s.visit_detail_concept_id,
    s.visit_detail_start_date,
    s.visit_detail_start_datetime,
    s.visit_detail_end_date,
    s.visit_detail_end_datetime,
    s.visit_detail_type_concept_id,
    NULL AS provider_id,
    NULL AS care_site_id,
    s.visit_detail_source_value,
    s.visit_detail_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    NULL AS preceding_visit_detail_id,
    NULL AS parent_visit_detail_id,
    0 AS visit_occurrence_id
  FROM _exponent.omop_silver.visit_detail s
  JOIN _exponent.omop_mapping.source_to_visit_detail svd
    ON svd.visit_detail_source_value = s.visit_detail_source_key
   AND svd.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.visit_detail_id = src.visit_detail_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                      = src.person_id,
  gold.visit_detail_concept_id        = src.visit_detail_concept_id,
  gold.visit_detail_start_date        = src.visit_detail_start_date,
  gold.visit_detail_start_datetime    = src.visit_detail_start_datetime,
  gold.visit_detail_end_date          = src.visit_detail_end_date,
  gold.visit_detail_end_datetime      = src.visit_detail_end_datetime,
  gold.visit_detail_type_concept_id   = src.visit_detail_type_concept_id,
  gold.provider_id                    = src.provider_id,
  gold.care_site_id                   = src.care_site_id,
  gold.visit_detail_source_value      = src.visit_detail_source_value,
  gold.visit_detail_source_concept_id = src.visit_detail_source_concept_id,
  gold.admitted_from_concept_id       = src.admitted_from_concept_id,
  gold.admitted_from_source_value     = src.admitted_from_source_value,
  gold.discharged_to_concept_id       = src.discharged_to_concept_id,
  gold.discharged_to_source_value     = src.discharged_to_source_value,
  gold.preceding_visit_detail_id      = src.preceding_visit_detail_id,
  gold.parent_visit_detail_id         = src.parent_visit_detail_id,
  gold.visit_occurrence_id            = src.visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_detail_id,
  person_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_value,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id,
  visit_occurrence_id
)
VALUES (
  src.visit_detail_id,
  src.person_id,
  src.visit_detail_concept_id,
  src.visit_detail_start_date,
  src.visit_detail_start_datetime,
  src.visit_detail_end_date,
  src.visit_detail_end_datetime,
  src.visit_detail_type_concept_id,
  src.provider_id,
  src.care_site_id,
  src.visit_detail_source_value,
  src.visit_detail_source_concept_id,
  src.admitted_from_concept_id,
  src.admitted_from_source_value,
  src.discharged_to_concept_id,
  src.discharged_to_source_value,
  src.preceding_visit_detail_id,
  src.parent_visit_detail_id,
  src.visit_occurrence_id
);

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.visit_detail WHERE source_system = 'allscripts_scm'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_visit_detail WHERE source_system = 'allscripts_scm'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.visit_detail

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.visit_detail
# LIMIT 10